In [24]:
# 모듈 import
from datetime import datetime
from annotated_types import Gt
from pydantic import BaseModel, PositiveInt, ValidationError
from typing import Annotated, Literal

In [25]:
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
# 클래스 선언부
class User(BaseModel):
  id: int
  name: str = 'John Doe'
  signup_ts: datetime | None
  tastes: dict[str, PositiveInt]

In [27]:
# 데이터
external_data = {
  'id': 123,
  'signup_ts': '2019-06-01 12:22',
  'tastes': {
      'wine': 9,
        b'cheese': 7,      # bytes → str 자동 변환
        'cabbage': '1',    # str '1' → int 1 자동 변환
  },
}

In [28]:
#User 오브젝트 생성
user = User(**external_data)
# “Pydantic에서는 외부 데이터를 넣는 순간 검증하고 객체까지 바로 생성
# User(...)
# → 우리가 만든 Pydantic User 모델에 데이터를 넣는다.
# **external_data
# **는 딕셔너리를 키워드 인자 형태로 풀어준다(unpacking)는 뜻

In [29]:
try:
    print(x)  # x 출력 시도
except:
    print("An exception occurred")  # 오류 발생 시 실행

An exception occurred


In [30]:
print(user.id)
print(user.signup_ts)
print(user.tastes)

#  'cabbage': '1' 숫자로 자동 형변환

123
2019-06-01 12:22:00
{'wine': 9, 'cheese': 7, 'cabbage': 1}


In [31]:
try:
    print(x)  # 실행 시도
except:
    print("Something went wrong")  # 오류 발생 시 실행
finally:
    print("The 'try except' is finished")  # 오류 여부와 상관없이 항상 실행

Something went wrong
The 'try except' is finished


In [32]:
try:
    f = open("demofile.txt")  # 파일 열기 시도

    try:
        f.write("Lorum Ipsum")  # 파일 쓰기 시도
    except:
        print("Something went wrong when writing to the file")  # 쓰기 오류 처리
    finally:
        f.close()  # 오류 여부와 상관없이 파일 닫기

except:
    print("Something went wrong when opening the file")  # 파일 열기 오류 처리

Something went wrong when opening the file


In [33]:
external_data2 = {
      'id': 'not an int',
      'tastes': {}
}

In [34]:
try:
    User(**external_data2)  # Pydantic 데이터 검증 시도
except ValidationError as e:
    print(e.errors())       # 검증 오류 발생 시 상세 내용 출력

# external_data2 딕셔너리에는 signup_ts 키가 아예 없었기 때문에,
# Pydantic은 이 필드가 누락되었다고 판단하여 missing 오류를 발생


[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [35]:
# Why use Pydantic Validation?
# Type hints powering schema validation

class Fruit(BaseModel):
    name: str
    # 문자열

    color: Literal['red', 'green']
    # 'red' 또는 'green'만 허용

    weight: Annotated[float, Gt(0)]
    # float 타입 + 0보다 큰 값만 허용

    bazam: dict[str, list[tuple[int, bool, float]]]
    # dict → list → tuple 형태의 중첩 데이터 타입 검증

In [36]:
print(
    Fruit(
        name='Apple',
        color='red',
        weight=4.2,
        bazam={'foobar': [(1, True, 0.1)]},
        # 리스트 안에 튜플: (정수, 불리언, 실수)
    )
)
# Fruit 객체 생성 → Pydantic 검증 → 통과하면 출력

name='Apple' color='red' weight=4.2 bazam={'foobar': [(1, True, 0.1)]}


In [37]:
class Meeting(BaseModel):
    when: datetime
    where: bytes
    why: str = 'No idea'

In [38]:
m = Meeting(
    when='2020-01-01T12:00',  # 날짜 입력
    where='home'               # 장소 입력
)                              # why는 안 넣음 → 기본값 'No idea'

In [39]:
print(m.model_dump(exclude_unset=True))
# 내가 직접 입력하지 않은 값은 제외 → why 빠짐
# 결과는 Python 딕셔너리(dict)

# {'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}

{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}


In [40]:
print(m.model_dump(exclude={'where'}, mode='json'))
# exclude={'where'} → where 빼기
# mode='json' → datetime 등을 JSON에서 사용 가능한 형태로 변환
# 결과는 여전히 Python 딕셔너리(dict)

# {'when': '2020-01-01T12:00:00', 'why': 'No idea'}

{'when': '2020-01-01T12:00:00', 'why': 'No idea'}


In [41]:
print(m.model_dump_json(exclude_defaults=True))
# 기본값과 같은 값은 제외 → why='No idea' 빠짐
# model_dump_json() → JSON 문자열로 변환

# {"when":"2020-01-01T12:00:00","where":"home"}
# dump = 꺼내기, exclude = 빼기

{"when":"2020-01-01T12:00:00","where":"home"}
